# 非线性方程(组)数值解  
方程求解一直是数学中的核心问题之一，然而，即使是对千形如
$$ \sum_{i=0}^{n} a_ix^i = 0 $$
这样的代数方程，当 $ n \geqslant 5$ 时也没有统一的求根公式．
在实际应用中，方程的数值解往往就可以满足工程及计算的需要了．这里介绍
两种较为常用的方程数值解法：二分法、牛顿迭代法．读者需要了解这些方法的使
用条件


## 二分法求根

若 $f(x)\in C[a,b]$($[a,b]$上的连续函数) 且$f(a)f(b)<0$，则由介值定理，$\exists\,c\in[a,b]$，使得$f(c)=0$.这时，可以使用二分法对方称求根.  

1. 令 $a_0 = a$, $b_0 = b$, $n = 0$.  
2. 令 $c_n = \frac{a_n + b_n}{2}$.  
3. 若 $|f(c_n)| < \varepsilon$，则算法停止，输出 $c_n$.  
4. 若 $f(a_n)f(c_n) < 0$，则 $a_{n+1} \leftarrow c_n$, $b_{n+1} \leftarrow b_n$；否则，$a_{n+1} \leftarrow a_n$, $b_{n+1} \leftarrow c_n$.  
5. $n \leftarrow n + 1$，转至 (2).

采用二分法对方程求根时，第 $n$ 次迭代对应的区间长度为 $\frac{b - a}{2^n}$（区间长度每次砍掉一半），收敛速度是较快的.

## 牛顿迭代求根
若 $f(x) \in C^2[a, b]$（$[a, b]$ 上的二阶连续可微函数），$f(a)f(b) < 0$ 且 $f'(x)$ 在 $[a, b]$ 上不变号，则方程 $f(x) = 0$ 在 $[a, b]$ 内必然存在某个根 $x^*$。设 $x_0$ 是 $x^*$ 附近的点，则根据泰勒展开式（一阶截断）有：

$$
0 = f(x^*) = f(x_0) + f'(x_0)(x^* - x_0) + \frac{f''(\xi_0)}{2}(x^* - x_0)^2 \tag{3.7}
$$

令 $x_1 = x_0 - \frac{f(x_0)}{f'(x_0)}$，那么：

$$
\begin{aligned}
x^* - x_1 &= x^* - x_0 + \frac{f(x_0)}{f'(x_0)} \\
&\stackrel{(3.7)}{=} \frac{-f(x_0) - \frac{f''(\xi_0)}{2}(x^* - x_0)^2}{f'(x_0)} + \frac{f(x_0)}{f'(x_0)} \\
&= -\frac{f''(\xi_0)}{2f'(x_0)}(x^* - x_0)^2
\end{aligned}
$$

即：

$$
\frac{x^* - x_1}{(x^* - x_0)^2} = -\frac{f''(\xi_0)}{2f'(x_0)}
$$（可以看出俯冲速度二次方，二阶收敛）

同样，对每个 $i$，若令：

$$
x_{i+1} = x_i - \frac{f(x_i)}{f'(x_i)} \tag{3.8}
$$

则有：

$$
\frac{x^* - x_{i+1}}{(x^* - x_i)^2} = -\frac{f''(\xi_i)}{2f'(x_i)}
$$

若存在 $\( M = \max\limits_{x \in [a, b]} |f''(x)| / \min\limits_{x \in [a, b]} |f'(x)| \)$，则：

$$
\frac{|x^* - x_{i+1}|}{|x^* - x_i|} \leq \frac{M}{2} |x^* - x_i|
$$

这说明该序列能够以较快的速度收敛于 $\( x^* \)$。该方法称为牛顿迭代法。

## 用 SciPy 工具库求解非线性方程（方程组）  
## scipy.optimize.fsolve(func, x0, args=(), fprime=None, full_output=0, 
## xtol=1.49012e-08, maxfev=0, ...)

fsolve 记住 func, x0, args, xtol, fprime 五个就够用了

| 参数               | 说明                               |
| ---------------- | -------------------------------- |
| **func**         | 目标函数 `f(x)`，要求 `f(x)=0`。         |
| **x0**           | 初值（标量或一维数组）。                     |
| **args**         | 传给 `func` 的额外参数元组 `f(x, *args)`。 |
| **fprime**       | 可选：提供雅可比矩阵/梯度函数，可加速收敛。           |
| **xtol**         | 位置容差，默认约 1.5e-8。                 |
| **maxfev**       | 最大函数调用次数，0 表示自动。                 |
| **full\_output** | 设为 1 返回完整结果字典（根、信息、迭代次数等）。       |


求方程 $x^3 + 1.1x^2 + 0.9x - 1.4 = 0$实根的近似值，使误差不超过$10^{-6}$.要求使用三种方法

很容易可以知道方程在区间(0, 1)上有一个零点

二分法  
它只保证 根落在区间内，不保证函数值很小。  
如果根靠近区间端点，即使区间已经很窄，f(c) 仍可能较大。  
因此用 函数值误差 |f(c)|<eps 更直观，也与“求零点”的目标一致。  
牛顿法  
每一步都用 切线近似，理论上一旦接近根，函数值就会迅速变小；  
但数值上可能出现 导数极小 或 根附近非常平坦 的情况，导致 f(x) 很小，而 x 离真根仍有一段距离。  
因此用 位置误差 |x1−x0|<eps 更稳健，防止提前停机。  
（当然也可以双条件：|x1−x0|<eps 且 |f(x1)|<eps，后者作为可选检查。）

In [11]:
import numpy as np
from scipy.optimize import fsolve

def binary_search(f, eps, a, b):
    c = (a + b) / 2
    while np.abs(f(c))>eps:  # 即f(c)与0的误差大于 eps
        if f(a)*f(c) < 0:  # 零点在左半区间
            b = c  # 后根据 c=(a+b)/2 重新确定 c
        else:  # 零点在右半区间
            a = c
        c = (a+b) / 2  # 重新设置 c 点，否则会出错，陷入死循环，该语句应在循环内，每循环一次，c 值都要更新
    return c

def newton_iter(f, eps, x0, dx=1E-8):  # 三个位置参数，一个默认参数
    def diff(f, dx = dx):  # 一个位置参数，一个默认参数
        return lambda x: (f(x+dx)-f(x-dx))/(2*dx)  # 中心差分（二阶精度）的一阶导数，返回的是一个---函数对象---
    df = diff(f, dx)  # df 为 f 的一阶导数
    # 切线俯冲，快速逼近零点，先写出切线方程，再令 y=0 求出 x1
    x1 = x0 - f(x0) / df(x0)
    while np.abs(x1-x0) >= eps:
        x0 = x1  # 相当于此时x0=x1，保证x0，x1只差一次迭代，必须要先更新 x0，否则x1更新的值会赋值给 x0，即 x0 = x2
        x1 = x1 - f(x1) / df(x1)  # 相当于此时x1=x2，以此类推
    return x1

f = lambda x: x**3+1.1*x**2+0.9*x-1.4
print(f"二分法求得的根为：{binary_search(f, 1E-6, 0, 1)}")
print(f"牛顿迭代法求得的根为：{newton_iter(f, 1E-6, 0)}")
print(f"调用SciPy求得的根为：{fsolve(f, 0)}")

二分法求得的根为：0.6706571578979492
牛顿迭代法求得的根为：0.6706573107258097
调用SciPy求得的根为：[0.67065731]


## 用 fsolve 求非线性方程组的数值解

求下列非线性方程组的数值解（下面已化好）

### 先将非齐次方程组化成线性，使用 fsolve 时，f 是等式各行左边的式子组成的类似列表的函数

fsolve(f, xo)  

这里x0为数组（维度与未知数相同），这里x0的选择不是随机的，其选择至关重要

如何选初值？  
物理/工程背景：根据实际问题给出合理估计。  
粗略扫描：在变量范围内随机/网格采样，找使 ‖f(x)‖ 最小的点作为初值。  
简单尝试：先用一维方法（如 np.roots、brentq）估计部分变量，再扩展。  

$$
\begin{cases}
5x_2 + 3 = 0, \\
4x_1^2 - 2\sin(x_2x_3) = 0, \\
x_2x_3 - 1.5 = 0.
\end{cases}
$$

In [13]:
import numpy as np
from scipy.optimize import fsolve  # optimize 优化
f = lambda x: [5*x[1]+3, 4*x[0]**2-2*np.sin(x[1]*x[2]), x[1]*x[2]-1.5]
print(f"result = {fsolve(f, [1, 1, 1])}")

result = [-0.70622057 -0.6        -2.5       ]


### 选取初值的案例  
可以看到初值为[0, 0, 0]时结果错误  
ier 是 Info Return Code 的缩写，表示算法终止的状态码：  
ier = 1 表示成功收敛到根。  
其他值表示不同类型的失败或警告（如 ier = 2, 3, 4, 5 对应迭代次数超限、精度不足等）。  
fev 是 Function Evaluations 的缩写，表示算法过程中调用目标函数 f(x) 的总次数，反映计算成本。  

In [14]:
import numpy as np
from scipy.optimize import fsolve

f = lambda x: [5*x[1]+3,
               4*x[0]**2-2*np.sin(x[1]*x[2]),
               x[1]*x[2]-1.5]

for guess in [[1,1,1], [0,0,0], [-1,1,1], [2,2,2]]:
    root, info, ier, _ = fsolve(f, guess, full_output=True)
    print(f"guess={guess}  ->  root={root}  |  ier={ier}  |  fev={info['nfev']}")

guess=[1, 1, 1]  ->  root=[-0.70622057 -0.6        -2.5       ]  |  ier=1  |  fev=22
guess=[0, 0, 0]  ->  root=[0. 0. 0.]  |  ier=5  |  fev=17
guess=[-1, 1, 1]  ->  root=[ 0.70622057 -0.6        -2.5       ]  |  ier=1  |  fev=22
guess=[2, 2, 2]  ->  root=[ 0.70622057 -0.6        -2.5       ]  |  ier=1  |  fev=39


上面的程序使用的是匿名函数，但是Python的下标从0开始，不太方便。下面利用函数定义方程组

In [18]:
from numpy import sin
from scipy.optimize import fsolve
def Pfun(x):
    x1, x2, x3 = x.tolist()  # x 转换成列表
    return 5*x2+3, 4*x1**2-2*sin(x2*x3), x2*x3-1.5
print(f"result = {fsolve(Pfun, [1.0, 1.0, 1.0])}")

result = [-0.70622057 -0.6        -2.5       ]
